In [ ]:
import socket
import json
import threading
import time
import random
import sys
import ssl

HOST = '127.0.0.1'
PORT = 9999

JOBS_PER_CLIENT = 5

In [ ]:
#Helpers
def send_msg(sock, msg: dict):
    data = json.dumps(msg) + '\n'
    sock.sendall(data.encode('utf-8'))


def recv_msg(sock) -> dict:
    buffer = ''
    while True:
        chunk = sock.recv(4096).decode('utf-8')
        if not chunk:
            raise ConnectionError('Server closed connection')
        buffer += chunk
        if '\n' in buffer:
            line, buffer = buffer.split('\n', 1)
            return json.loads(line)

In [ ]:
#Submit job & Poll for completion
def submit_job(payload: dict):
    # Setup Client SSL Context
    context = ssl.create_default_context()
    context.check_hostname = False
    context.verify_mode = ssl.CERT_NONE

    # 1. Submit the job securely
    try:
        raw_s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s = context.wrap_socket(raw_s, server_hostname=HOST)
        s.connect((HOST, PORT))
        
        send_msg(s, {'type': 'submit', 'payload': payload})
        resp = recv_msg(s)
        job_id = resp.get('job_id')
        s.close()
    except Exception as e:
        print("Submission error:", e)
        return None

    if not job_id: 
        return None

    # 2. Poll the server securely until the job is complete
    while True:
        time.sleep(0.5)
        try:
            raw_s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s = context.wrap_socket(raw_s, server_hostname=HOST)
            s.connect((HOST, PORT))
            
            send_msg(s, {'type': 'status', 'job_id': job_id})
            resp = recv_msg(s)
            s.close()
            
            if resp.get('status') == 'done':
                return resp.get('result')
        except Exception:
            pass

In [ ]:
#Simulated Client
def simulated_client(client_id):
    tasks = ['math', 'fibonacci', 'sleep']
    operations = ['add', 'subtract', 'multiply', 'divide']
    symbols = {'add': '+', 'subtract': '-', 'multiply': '*', 'divide': '/'}

    for _ in range(JOBS_PER_CLIENT):
        task_type = random.choice(tasks)
        
        if task_type == 'math':
            op = random.choice(operations)
            payload = {
                "task": "math",
                "operation": op,
                "a": random.randint(1, 100),
                "b": random.randint(1, 100)
            }
        elif task_type == 'fibonacci':
            payload = {
                "task": "fibonacci",
                "n": random.randint(10, 35)
            }
        elif task_type == 'sleep':
            payload = {
                "task": "sleep",
                "duration": random.randint(1, 3)
            }
            
        result = submit_job(payload)
        
        if result is not None:
            if task_type == 'math':
                sym = symbols[payload['operation']]
                print(f"[Client {client_id}] {payload['a']} {sym} {payload['b']} = {result}")
            elif task_type == 'fibonacci':
                print(f"[Client {client_id}] fib({payload['n']}) = {result}")
            elif task_type == 'sleep':
                print(f"[Client {client_id}] slept for {payload['duration']}s -> {result}")
        else:
            print(f"[Client {client_id}] Job failed or timed out.")

In [ ]:
#Run Simulation
def run_simulation(num_clients):
    threads = []

    print(f"\nRunning simulation with {num_clients} clients")

    start_time = time.time()

    for i in range(num_clients):
        t = threading.Thread(target=simulated_client, args=(i,))
        t.start()
        threads.append(t)

    for t in threads:
        t.join()

    end_time = time.time()

    total_jobs = num_clients * JOBS_PER_CLIENT
    duration = end_time - start_time
    throughput = total_jobs / duration if duration > 0 else 0

    print("\n=== Load Test Result ===")
    print(f"Clients: {num_clients}")
    print(f"Total Jobs: {total_jobs}")
    print(f"Time Taken: {duration:.2f}s")
    print(f"Throughput: {throughput:.2f} jobs/sec")

In [ ]:
run_simulation(5)